# 10 — Génération des scalogrammes WESAD

Ce notebook construit une représentation temps-fréquence sans modifier les fenêtres de 30 secondes produites par le notebook 01. La séparation par sujet précède donc la normalisation temporelle, la CWT et toute normalisation des scalogrammes. Les statistiques du test ne participent à aucune décision.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import *
from src.helpers import count_parameters, set_seed

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Périphérique sélectionné :", device)
print("CUDA disponible :", torch.cuda.is_available())

Périphérique sélectionné : cuda
CUDA disponible : True


## Protocole de transformation

Pour chaque fenêtre standardisée avec les statistiques des sujets d'entraînement, trois signaux sont retenus : BVP, EDA et norme de l'accélération

\[
a(t)=\sqrt{a_x(t)^2+a_y(t)^2+a_z(t)^2}.
\]

La chaîne est : fenêtre temporelle standardisée → CWT de Morlet → module des coefficients → `log1p` → interpolation bilinéaire 64 × 64 → normalisation par canal apprise uniquement sur l'entraînement.

Employer les mêmes échelles 1 à 64 pour les trois modalités est une hypothèse simplificatrice : BVP, EDA et accélération n'occupent pas les mêmes bandes fréquentielles. Ces paramètres constituent ici une référence fixée sur validation, jamais sur test.

In [2]:
from src.scalograms import (
    build_scalogram_artifacts,
    validate_subject_splits,
    window_to_log_scalogram,
)

validate_subject_splits()
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert set(SPLIT_SUBJECTS[left]).isdisjoint(SPLIT_SUBJECTS[right])
print(SPLIT_SUBJECTS)

{'train': ['S3', 'S4', 'S6', 'S7', 'S8', 'S9', 'S10', 'S13', 'S16', 'S17'], 'validation': ['S5', 'S15'], 'test': ['S2', 'S11', 'S14']}


## Contrôle synthétique de forme et de valeurs

In [3]:
synthetic_window = np.random.default_rng(RANDOM_SEED).normal(size=(960, 6)).astype(np.float32)
synthetic_map = window_to_log_scalogram(synthetic_window)
assert synthetic_map.shape == (3, 64, 64)
assert torch.isfinite(synthetic_map).all()
print("Forme d'un scalogramme :", tuple(synthetic_map.shape))

Forme d'un scalogramme : (3, 64, 64)


## Génération complète

La cellule suivante est volontairement protégée pour éviter d'écraser un cache volumineux. Mettre `RUN_SCALOGRAM_GENERATION=True` après exécution du notebook 01. Le fichier de métadonnées sauvegarde les sujets, le nom de l'ondelette, les échelles, la fréquence, l'ordre des canaux, la taille, la graine et les moments d'entraînement.

In [4]:
RUN_SCALOGRAM_GENERATION = False
scalogram_dir = PROJECT_ROOT / "data" / "processed" / "scalograms"
cache_ready = all(
    (scalogram_dir / f"X_{split}.pt").exists()
    and (scalogram_dir / f"y_{split}.pt").exists()
    for split in SPLIT_SUBJECTS
)

if RUN_SCALOGRAM_GENERATION:
    scalogram_metadata = build_scalogram_artifacts(PROJECT_ROOT, overwrite=False)
    display(pd.Series(scalogram_metadata, name="valeur"))
elif cache_ready:
    metadata_path = PROJECT_ROOT / "artifacts" / "preprocessing" / "scalograms" / "scalogram_metadata.json"
    with metadata_path.open(encoding="utf-8") as handle:
        scalogram_metadata = json.load(handle)
    print("Cache de scalogrammes déjà disponible ; aucune régénération nécessaire.")
    display(pd.Series(scalogram_metadata, name="valeur"))
else:
    print("Non exécuté : activer RUN_SCALOGRAM_GENERATION pour générer les données réelles.")

Cache de scalogrammes déjà disponible ; aucune régénération nécessaire.


subject_lists                          {'train': ['S3', 'S4', 'S6', 'S7', 'S8', 'S9',...
scalogram_shape                                                              [3, 64, 64]
wavelet                                                                             morl
scales                                 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...
sampling_rate_hz                                                                      32
sampling_period_seconds                                                          0.03125
resize_dimensions                                                               [64, 64]
channel_order                                                  [BVP, EDA, ACC_magnitude]
normalization_means                    [0.5726479551899668, 0.04516453070809788, 0.13...
normalization_standard_deviations      [0.5993747743130065, 0.13824641402697635, 0.24...
normalization_fitted_on                                                            train
time_domain_normaliza

## Validation du cache sauvegardé

In [5]:
scalogram_dir = PROJECT_ROOT / "data" / "processed" / "scalograms"
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"
if (scalogram_dir / "X_train.pt").exists():
    for split in SPLIT_SUBJECTS:
        X = torch.load(scalogram_dir / f"X_{split}.pt", map_location="cpu", weights_only=True)
        y = torch.load(scalogram_dir / f"y_{split}.pt", map_location="cpu", weights_only=True)
        meta = pd.read_csv(metadata_dir / f"windows_{split}.csv")
        assert X.shape[1:] == (3, 64, 64) and len(X) == len(y) == len(meta)
        assert torch.isfinite(X).all() and set(y.int().tolist()) <= {0, 1}
        assert set(meta.subject_id) <= set(SPLIT_SUBJECTS[split])
        print(split, tuple(X.shape))
else:
    print("Cache absent : exécuter la cellule de génération complète.")

train (1429, 3, 64, 64)
validation (287, 3, 64, 64)


test (426, 3, 64, 64)
